In [ ]:
import psycopg2

ESTABLISH CONNECTION WITH POSTGRE SERVER AND IMPORT LIBRARIES

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")


In [ ]:
#didn't uv add seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import glob
from io import BytesIO

Source - National Bueau of Statistics Nigeria

The inconsistent structure of the files had me cleaning them in batches

In [ ]:
Jan = pd.read_excel("Selected_food_table_Jan25.xlsx")
Jan.head()

In [ ]:
Feb = pd.read_excel("Selected_food_table_Feb25.xlsx")
Feb.head()

In [ ]:
Mar = pd.read_excel("Selected_food_table_Mar25.xlsx")
Mar.head()

In [ ]:
Apr = pd.read_excel("Selected_food_table_Apr25.xlsx")
Apr.head()

In [ ]:
#MAKE SURE PYTHON CAN SEE YOUR ZIPs
import glob
import os

print(os.getcwd())

zip_files = glob.glob("*.zip")

print(zip_files)

In [ ]:
#inspect what is inside the zips
for zip_path in zip_files:

    with zipfile.ZipFile(zip_path, 'r') as z:

        print(zip_path)
        print(z.namelist())
        print("------")

In [ ]:
zip_files = glob.glob("*.zip")

# separate normal files from Jan26
normal_zips = [z for z in zip_files if "Jan26" not in z]
jan26_zip = [z for z in zip_files if "Jan26" in z]

print("Normal:", normal_zips)
print("Jan26:", jan26_zip)

In [ ]:
normal_dfs = []
for zip_path in normal_zips:
    with zipfile.ZipFile(zip_path, 'r') as z:
        excel_files = [
            f for f in z.namelist()
            if f.lower().endswith(".xlsx")
        ]
        for file_name in excel_files:
            with z.open(file_name) as f:
                excel_data = BytesIO(f.read())
                df = pd.read_excel(excel_data)
                print(zip_path, "->", df.shape[0], "rows")
                # turn column names into first row
                header_row = pd.DataFrame([df.columns])
                # replace columns with generic positions
                df.columns = range(df.shape[1])
                header_row.columns = range(header_row.shape[1])
                # stack header row on top
                df = pd.concat([header_row, df], ignore_index=True)
                if "Oct25" in zip_path: 
                    date = "2025-10"
                elif "Nov_25" in zip_path:
                    date = "2025-11"
                elif "Dec_25" in zip_path:
                    date = "2025-12"
                elif "July_2025" in zip_path:
                    date = "2025-07"
                elif "June_2025" in zip_path:
                    date = "2025-06"
                elif "May_2025" in zip_path:
                    date = "2025-05"
                elif "Aug_2025" in zip_path:
                    date = "2025-08"
                elif "Sept_2025" in zip_path:
                    date = "2025-09"    
                df["date"] = date
                df["source_file"] = zip_path
                normal_dfs.append(df)
normal_df = pd.concat(normal_dfs, ignore_index=True)
print(normal_df.shape)

In [ ]:
normal_df.head(47)

In [ ]:
normal_df.tail(44)

You confirmed:

each block = 42 data rows
plus 1 header row at the top of each block
so total cycle = 43 rows per block

So the repeating pattern is:

row 0 → header
rows 1–42 → data
row 43 → header again
rows 44–85 → data
row 86 → header again
…and so on

So yes: your cycle length is 43 rows.

In [ ]:
jan26_dfs = []

for zip_path in jan26_zip:

    with zipfile.ZipFile(zip_path, 'r') as z:

        excel_files = [
            f for f in z.namelist()
            if f.lower().endswith(".xlsx")
        ]

        for file_name in excel_files:

            with z.open(file_name) as f:

                excel_data = BytesIO(f.read())

                jan26_df = pd.read_excel(excel_data)

                jan26_df["source_file"] = zip_path

                jan26_df["date"] = "2026-01"
                
                jan26_dfs.append(jan26_df)

jan26_final = pd.concat(jan26_dfs, ignore_index=True)

print(jan26_final.shape)

In [ ]:
final_df = pd.concat(
    [normal_df, jan26_final],
    ignore_index=True
)

print(final_df.shape)

In [ ]:
Jan["date"] = "2025-01"
Jan["source_file"] = "food-inflation-project"

In [ ]:
Jan.head()

In [ ]:
#check whcih food row prices are missing
Jan["Average of Jan-24"].isna

If we dropped rows that had 4 of the 6 (8 columns in total) important columns missing (except highest and lowest), we may end up dropping a row (eg rice) in df1, and keeping it in df2, therefore loosing timeseries consistency. So instead, we'll keep the rows where item labels exist and at least one important price (that is, 2 of the important rows) 

In [ ]:
price_cols = [
    "Average of Jan-24",
    "Average of Dec-24",
    "Average of Jan-25",
]

JanCopy = Jan.copy()

# keep rows where Item Label exists
# AND at least one price column has data

JanCopy = JanCopy[
    (JanCopy["Item Label"].notna()) &
    (JanCopy[price_cols].notna().any(axis=1))
]

JanCopy.head()

In [ ]:
#How many rows still have ALL price columns missing?
JanCopy[price_cols].isna().all(axis=1).sum()

To upscale dropping rows that have most of their important columns missing, we
use columns positions as the names change per dataframe.
    we use .iloc

In [ ]:
rem_excel_dfs = [Feb, Mar, Apr]

months = ["2025-02", "2025-03", "2025-04"]

cleaned_rem_excel_dfs = []

for df, month in zip(rem_excel_dfs, months):

    df = df[df.iloc[:, 0].notna()]

    df = df[df.iloc[:, 1:4].notna().any(axis=1)]

    df["date"] = month

    df["source_file"] = "food-inflation-project"

    cleaned_rem_excel_dfs.append(df)
    #meaning: all rows, column 1 to 3. 4 excluded because 
#python slicing stops before the endpoint
    

In [ ]:
df.iloc[:, 1:4].isna().all(axis=1).sum()

In [ ]:
#print all the dfs independently. you can't use .head() cause cleaned_rem... 
#is a cumulation of data frames
cleaned_rem_excel_dfs[0].head()

Time to clean the zips

In [ ]:
normal_df.head()

In [ ]:
normal_df["block_id"] = normal_df.index // 43 #0-42 is report block 1
#43-85 is report block 2, so you can call them
normal_df["row_in_block"] = normal_df.index % 43 #index 0 is 0 row_in_block,
#index 43 is 0 again for row_in_block

In [ ]:
#shows one full report block
normal_df[normal_df["block_id"] == 0].head(43)

In [ ]:
normal_df = normal_df.replace(["####", "###", "N/A", "-", ""], np.nan)

In [ ]:
normal_df.shape

In [ ]:
is_header = normal_df[0] == "Item Label"

In [ ]:
price_cols = [1, 2, 3]

is_header = normal_df[0] == "Item Label"

# numeric conversion ONLY for real data rows
normal_df.loc[~is_header, price_cols] = (
    normal_df.loc[~is_header, price_cols]
    .apply(pd.to_numeric, errors="coerce")
)

# keep headers + valid rows
valid_rows = normal_df.loc[~is_header, price_cols].notna().any(axis=1)

clean_normal_df = normal_df.loc[is_header | valid_rows]

In [ ]:
clean_normal_df.shape

In [ ]:
jan26_final = jan26_final.replace(["####", "###", "N/A", "-", ""], np.nan)


In [ ]:
jan26_final.head()

Its clean 

Time to push to postgre. Pick the tables. First with all the 4 excel files. Then January26. Then the zips

In [ ]:
conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)

cur = conn.cursor()
print("Connected Successfully")

In [ ]:
conn.rollback()

In [ ]:
#this code may push all the rows lol so how do you cherry pick which dfs to pick since we wanna start with jan26?
cur.execute("""
CREATE TABLE IF NOT EXISTS food_prices (
    item TEXT,
    avg_price FLOAT,
    highest TEXT,
    lowest TEXT,
    date TEXT,
    source_file TEXT
    );
    """)


In [ ]:
conn.commit()

In [ ]:
JanCopy.head()

In [ ]:
JanCopy.columns.str.strip().str.lower()
JanCopy = JanCopy.rename(columns={
    "Item Label": "item",
    "Average of Jan-25": "avg_price",
    "Highest":"highest",
    "Lowest":"lowest",
})

In [ ]:
for i, row in JanCopy.iterrows():
    cur.execute("""
    INSERT INTO food_prices (item, avg_price, highest, lowest, date, source_file)
    VALUES (%s, %s, %s, %s, %s, %s)
    """, (row["item"], row["avg_price"], row["highest"], row["lowest"], row["date"], row["source_file"]))

In [ ]:
for df in cleaned_rem_excel_dfs:

    cols = list(df.columns)

    cols[0] = "item"
    cols[3] = "avg_price"
    cols[6] = "highest"
    cols[7] = "lowest"

    df.columns = cols

In [ ]:
for df in cleaned_rem_excel_dfs:

    for i, row in df.iterrows():

        cur.execute("""
        INSERT INTO food_prices
        (item, avg_price, highest, lowest, date, source_file)

        VALUES (%s, %s, %s, %s, %s, %s)
        """,

        (
            row["item"],
            row["avg_price"],
            row["highest"],
            row["lowest"],
            row["date"],
            row["source_file"]
        ))

Now, to push the zips

Remove all repeated header rows
Rename columns once
Push to sql

In [ ]:
zip_push_df = normal_df[normal_df["row_in_block"] != 0]

In [ ]:
cols = list(zip_push_df.columns)

cols[0] = "item"
cols[3] = "avg_price"
cols[6] = "highest"
cols[7] = "lowest"

zip_push_df.columns = cols

In [ ]:
for i, row in zip_push_df.iterrows():

    cur.execute("""
    INSERT INTO food_prices
    (item, avg_price, highest, lowest, date, source_file)

    VALUES (%s, %s, %s, %s, %s, %s)
    """,

    (
        row["item"],
        row["avg_price"],
        row["highest"],
        row["lowest"],
        row["date"],
        row["source_file"]
    ))

To push jan26 now

In [ ]:
jan26_final
jan26_final.columns.str.strip().str.lower()
jan26_final = jan26_final.rename(columns={
    "Item Label": "item",
    "Average of January-26": "avg_price",
    "Highest":"highest",
    "Lowest":"lowest",
})

In [ ]:
for i, row in jan26_final.iterrows():
    cur.execute("""
    INSERT INTO food_prices (item, avg_price, highest, lowest, date, source_file)
    VALUES (%s, %s, %s, %s, %s, %s)
    """, (row["item"], row["avg_price"], row["highest"], row["lowest"], row["date"], row["source_file"]))

In [ ]:
conn.commit()